# Govern Your Agents: Budgets, Approval Gates, and Exact Replay with Pollard

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jemsbhai/devfestTampaPollard2026/blob/main/workshop.ipynb)

Hands-on notebook for the DevFest workshop at USF Tampa, Saturday, November 14, 2026.

You take a small refund agent, route its calls through [Pollard](https://github.com/jemsbhai/pollard), and make the runtime say no four ways, on purpose.

| # | Exercise | You cause | What it shows |
|---|---|---|---|
| 1 | Budget | A budget refusal | A refused step never runs; a rerun pays only for new steps |
| 2 | Gate | A policy refusal | An approval binds one exact action and is used once |
| 3 | Evidence | A failed verification | The tree is the audit record; an edit is detected |
| 4 | Replay | A replay miss | Strict replay serves the record and never goes live |

**How the exercises work.** An exercise cell stops at the first line that still holds a `todo(n, "code")` call. Replace the whole call with the code inside the quotes, then run the cell again. Under each of the first two exercises there is a folded Solution cell: if you fall behind, run it and carry on.

After the exercises come the same flow on a real model, a summary of the calls you used, and every example, recipe and experiment script from the Pollard repository.

## Setup

Run these three cells before the session starts. They need no API key.

In [ ]:
POLLARD_VERSION = "1.6.2"   # the pin for this workshop; 1.6.1 is the fallback

%pip install -q "pollard=={POLLARD_VERSION}"

import os
import subprocess
import sys

from IPython.display import HTML

# The helpers for this notebook live in workshop_kit.py. Outside the repository folder, fetch the repository.
if not os.path.exists("workshop_kit.py"):
    if not os.path.isdir("devfestTampaPollard2026"):
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/jemsbhai/devfestTampaPollard2026"], check=True)
    os.chdir("devfestTampaPollard2026")

from workshop_kit import *

import pollard
print("pollard", pollard.__version__, "in", os.getcwd())

In [ ]:
MODEL_PATH = "scripted"     # "scripted", "colab" or "endpoint"

model = choose_model(MODEL_PATH)
reset()                     # choosing a model path starts a fresh store: run this cell to start over
print("root ID for this room:", root_id_by_hand()[:8])

In [ ]:
ready = self_check()        # the four exercises on the scripted model, in a temporary store

The last cell should print `PASS: every check held on pollard 1.6.2`. If it does, you are ready.

| Model path | What answers | Needs |
|---|---|---|
| `scripted` | A deterministic stand-in that makes the same three decisions every time | Nothing. Everyone's node IDs match the slides |
| `colab` | A model from Colab's built-in library, `google.colab.ai` | A Colab runtime where that library is available to your account |
| `endpoint` | Any OpenAI-compatible endpoint | `OPENAI_API_KEY` and `WORKSHOP_MODEL`, plus `OPENAI_BASE_URL` for anything other than OpenAI, as Colab Secrets or in a local `.env` |

A real path that fails its one-call check falls back to `scripted` and says so. On a real model the node IDs differ from the slides after the first model call, because the model's reply is part of the next step's input.

To start over at any point, run the model cell again. It deletes the store.

## The agent, ungoverned

The agent is a loop: ask the model for one decision, run the tool it names, feed the result back, stop at a final answer. It has two tools. `lookup_order` reads an order. `issue_refund` returns money.

Run this cell twice.

In [ ]:
def ungoverned_agent():
    reset_counters()
    messages = first_messages()
    while True:
        payload = {"model": model.name, "messages": messages}
        reply = model.fn(payload)
        decision = parse_decision(reply["text"])
        messages = messages + [assistant_message(reply["text"])]
        if "final" in decision:
            print("  FINAL:", decision["final"])
            break
        name, args = decision["tool"], decision["args"]
        result = registry.get(name).handler(args)
        print(f"  tool {name} -> {describe(name, result)}")
        messages = messages + [tool_message(name, result)]
    print("  live calls:", live_calls())


ungoverned_agent()

You should see, both times:

<!-- expect:ungoverned -->
```text
  tool lookup_order -> status delivered
  tool issue_refund -> refund_id R-77, cents 4200
  FINAL: Refund of $42.00 issued for order A-1001.
  live calls: model 3, lookup_order 1, issue_refund 1
```

Nothing limited the loop, and nobody was asked before money moved. The second run called the model again and refunded again. Neither run left a record.

## Exercise 1: Budget

**Goal:** put both call sites behind the runtime and give the run a budget of two steps.

The loop below is the same loop inside a governed run. A governed call returns a node, and the node's `.result` is what the function returned, so the rest of the loop does not change.

1. Run the cell. It stops at TODO 1.
2. Replace both `todo(...)` calls with the code inside their quotes. Run the cell.
3. Run the cell again, unchanged.
4. Set `STEPS = 3` and run it a third time.

In [ ]:
STEPS = 2      # this exercise uses 2, then 3


def budgeted_agent(steps):
    reset_counters()
    with SQLiteStore(DB) as store:
        runtime = Runtime(store, registry=registry, mode="hybrid")
        try:
            with runtime.run(RUN_LABEL, budget=Budget(steps=steps)) as run:
                messages = first_messages()
                while True:
                    payload = {"model": model.name, "messages": messages}
                    reply = todo(1, "run.model_call(payload, fn=model.fn).result")
                    decision = parse_decision(reply["text"])
                    messages = messages + [assistant_message(reply["text"])]
                    if "final" in decision:
                        print("  FINAL:", decision["final"])
                        break
                    name, args = decision["tool"], decision["args"]
                    result = todo(2, "run.tool_call(name, args).result")
                    print(f"  tool {name} -> {describe(name, result)}")
                    messages = messages + [tool_message(name, result)]
        except BudgetExceeded as refusal:
            print(f"  REFUSED by budget: {refusal}\n  refusal node {refusal.refusal_id[:8]}")
        except PolicyViolation as refusal:
            print(f"  REFUSED by policy: {refusal}\n  refusal node {refusal.refusal_id[:8]}")
        except ValueError as problem:
            print(f"  STOPPED: {problem}")
    print("  live calls:", live_calls())


budgeted_agent(STEPS)

After step 2 you should see:

<!-- expect:ex1-first -->
```text
  tool lookup_order -> status delivered
  REFUSED by budget: budget exceeded for steps
  refusal node 86840b53
  live calls: model 1, lookup_order 1
```

The budget paid for one model call and the lookup. The next model call was refused before it ran, and the refusal is itself a node in the tree.

After step 3, the same store serves the two recorded steps. Nothing runs, and the refusal is the same node:

<!-- expect:ex1-again -->
```text
  tool lookup_order -> status delivered
  REFUSED by budget: budget exceeded for steps
  refusal node 86840b53
  live calls: none
```

After step 4, the larger budget pays for exactly one new step. The model asks for the refund, and the refund is refused before its handler runs:

<!-- expect:ex1-three -->
```text
  tool lookup_order -> status delivered
  REFUSED by budget: budget exceeded for steps
  refusal node 84a72a90
  live calls: model 1
```

Keep `STEPS` at 3 or below in this exercise. With a larger budget the refund would run, because nothing gates it yet. That is Exercise 2.

In [ ]:
#@title Solution for Exercise 1 (run it if you are stuck)

def budgeted_agent(steps):
    reset_counters()
    with SQLiteStore(DB) as store:
        runtime = Runtime(store, registry=registry, mode="hybrid")
        try:
            with runtime.run(RUN_LABEL, budget=Budget(steps=steps)) as run:
                messages = first_messages()
                while True:
                    payload = {"model": model.name, "messages": messages}
                    reply = run.model_call(payload, fn=model.fn).result
                    decision = parse_decision(reply["text"])
                    messages = messages + [assistant_message(reply["text"])]
                    if "final" in decision:
                        print("  FINAL:", decision["final"])
                        break
                    name, args = decision["tool"], decision["args"]
                    result = run.tool_call(name, args).result
                    print(f"  tool {name} -> {describe(name, result)}")
                    messages = messages + [tool_message(name, result)]
        except BudgetExceeded as refusal:
            print(f"  REFUSED by budget: {refusal}\n  refusal node {refusal.refusal_id[:8]}")
        except PolicyViolation as refusal:
            print(f"  REFUSED by policy: {refusal}\n  refusal node {refusal.refusal_id[:8]}")
        except ValueError as problem:
            print(f"  STOPPED: {problem}")
    print("  live calls:", live_calls())


for steps in (2, 2, 3):
    print(f"steps={steps}")
    budgeted_agent(steps)

## Exercise 2: Gate

**Goal:** money moves only after a named reviewer approves that exact call.

`ApprovalPolicy` refuses every action with side effects until the store holds an approval for it. `request_approval` records which call is waiting, and returns the request a reviewer decides on.

1. Replace both `todo(...)` calls and run the cell. The budget is now 8 steps.

In [ ]:
def gated_agent(steps, mode="hybrid", task=TASK, db=DB):
    replay = mode == "replay"                  # Exercise 4: in strict replay nothing may run,
    reg = make_registry(replay)                #   so every tool handler becomes a sentinel
    fn = sentinel if replay else model.fn      #   and so does the model function
    reset_counters()
    request = None
    with SQLiteStore(db, read_only=replay) as store:
        policies = todo(1, "[ApprovalPolicy(store)]")
        runtime = Runtime(store, registry=reg, policies=policies, mode=mode)
        try:
            with runtime.run(RUN_LABEL, budget=Budget(steps=steps)) as run:
                messages = first_messages(task)
                while True:
                    payload = {"model": model.name, "messages": messages}
                    reply = run.model_call(payload, fn=fn).result
                    decision = parse_decision(reply["text"])
                    messages = messages + [assistant_message(reply["text"])]
                    if "final" in decision:
                        print("  FINAL:", decision["final"])
                        break
                    name, args = decision["tool"], decision["args"]
                    if has_side_effects(name):
                        request = todo(2, "request_approval(run, name, args)")
                    result = run.tool_call(name, args).result
                    print(f"  tool {name} -> {describe(name, result)}")
                    messages = messages + [tool_message(name, result)]
        except BudgetExceeded as refusal:
            print(f"  REFUSED by budget: {refusal}\n  refusal node {refusal.refusal_id[:8]}")
        except PolicyViolation as refusal:
            print(f"  REFUSED by policy: {refusal}")
            print(f"  refusal node {refusal.refusal_id[:8]}, request {request.id[:8] if request else 'none'}")
        except MissingRecording as miss:
            print("  MissingRecording:", str(miss)[:60])
        except ValueError as problem:
            print(f"  STOPPED: {problem}")
    print("  live calls:", live_calls())
    return request


request = gated_agent(8)

The refund is refused again, this time by policy, and the refusal names the request that is waiting:

<!-- expect:ex2-first -->
```text
  tool lookup_order -> status delivered
  REFUSED by policy: durable approval is required for this action
  refusal node d4f027bc, request 6eab2bbe
  live calls: none
```

2. Approve the request. Put your own name in `REVIEWER` before you run this cell, because a request is decided once. Then replace the `todo(...)` call and run it.

In [ ]:
REVIEWER = "your-name"       # your own name: the reviewer is part of the record
assert request is not None, "run the cell above first: no approval request exists yet"

with SQLiteStore(DB) as store:
    try:
        decision = todo(3, "decide_approval(store, request, approved=True, reviewer=REVIEWER)")
        print("decision node", decision.id[:8], "recorded for request", request.id[:8], "by", REVIEWER)
    except IntegrityError as conflict:
        print("REFUSED:", conflict)

With the reviewer left as `your-name` the cell prints:

<!-- expect:ex2-approve -->
```text
decision node aae163fa recorded for request 6eab2bbe by your-name
```

Your decision node has a different ID if you used your own name: the reviewer is part of the node's content.

3. Run the agent again. Then run this same cell a second time.

In [ ]:
request = gated_agent(8)

The first run after the approval serves the recorded steps, runs the refund handler once and finishes:

<!-- expect:ex2-approved -->
```text
  tool lookup_order -> status delivered
  tool issue_refund -> refund_id R-77, cents 4200
  FINAL: Refund of $42.00 issued for order A-1001.
  live calls: model 1, issue_refund 1
```

The second run is refused, with no live calls. The approval was for one call, and that call has been made:

<!-- expect:ex2-spent -->
```text
  tool lookup_order -> status delivered
  REFUSED by policy: durable approval is required for this action
  refusal node d4f027bc, request 6eab2bbe
  live calls: none
```

In [ ]:
#@title Solution for Exercise 2 (run it if you are stuck)

def gated_agent(steps, mode="hybrid", task=TASK, db=DB):
    replay = mode == "replay"                  # Exercise 4: in strict replay nothing may run,
    reg = make_registry(replay)                #   so every tool handler becomes a sentinel
    fn = sentinel if replay else model.fn      #   and so does the model function
    reset_counters()
    request = None
    with SQLiteStore(db, read_only=replay) as store:
        policies = [ApprovalPolicy(store)]
        runtime = Runtime(store, registry=reg, policies=policies, mode=mode)
        try:
            with runtime.run(RUN_LABEL, budget=Budget(steps=steps)) as run:
                messages = first_messages(task)
                while True:
                    payload = {"model": model.name, "messages": messages}
                    reply = run.model_call(payload, fn=fn).result
                    decision = parse_decision(reply["text"])
                    messages = messages + [assistant_message(reply["text"])]
                    if "final" in decision:
                        print("  FINAL:", decision["final"])
                        break
                    name, args = decision["tool"], decision["args"]
                    if has_side_effects(name):
                        request = request_approval(run, name, args)
                    result = run.tool_call(name, args).result
                    print(f"  tool {name} -> {describe(name, result)}")
                    messages = messages + [tool_message(name, result)]
        except BudgetExceeded as refusal:
            print(f"  REFUSED by budget: {refusal}\n  refusal node {refusal.refusal_id[:8]}")
        except PolicyViolation as refusal:
            print(f"  REFUSED by policy: {refusal}")
            print(f"  refusal node {refusal.refusal_id[:8]}, request {request.id[:8] if request else 'none'}")
        except MissingRecording as miss:
            print("  MissingRecording:", str(miss)[:60])
        except ValueError as problem:
            print(f"  STOPPED: {problem}")
    print("  live calls:", live_calls())
    return request


print("first run")
request = gated_agent(8)
REVIEWER = globals().get("REVIEWER", "your-name")
if request is not None:
    with SQLiteStore(DB) as store:
        try:
            decision = decide_approval(store, request, approved=True, reviewer=REVIEWER)
            print("decision node", decision.id[:8], "recorded for request", request.id[:8], "by", REVIEWER)
        except IntegrityError as conflict:
            print("REFUSED:", conflict)
print("after the approval")
request = gated_agent(8)
print("one more run")
request = gated_agent(8)

## Exercise 3: Evidence

**Goal:** read the record that the last two exercises wrote, then show that an edit to it is detected.

Every step, refusal, request and decision is a node. A node's ID is a hash of its kind, its input and its parent's ID. Nothing to fill in here: run each cell and read what it prints.

In [ ]:
root = root_id_by_hand()               # the root ID is a hash of the run label alone
print("root", root[:8])
_ = pollard_cli("show", DB, root)      # in a terminal: pollard show workshop.db <root id>

The tree has twelve nodes: the root, three model calls, two tool calls, three refusals, and three notes (the approval request, its decision slot and the decision). Every `[REFUSED]` line is a step that did not run. If you ran the Exercise 1 cell again after finishing Exercise 2, you have an extra refusal or two: every refused attempt is recorded.

The next cell draws the same tree with every payload and result. Click a node to fold or unfold it.

In [ ]:
HTML(tree_html(payloads=True))

In [ ]:
_ = pollard_cli("report", DB, root)
print("verify exit code:", pollard_cli("verify", DB))
_ = pollard_cli("seal", DB, root)

`spent` is what executed. `avoided` is what your reruns were served from the record. `verify` recomputes every node ID and result digest. The seal is one digest over the whole tree: store it somewhere else, and you can show later that the record has not changed. Your seal differs from your neighbor's if your reviewer names differ.

Now edit the record. `tamper()` copies the store and changes the recorded refund in the copy from 4200 cents to 42.

In [ ]:
print(tamper())                        # writes tampered.db; workshop.db is not touched
print("verify exit code:", pollard_cli("verify", "tampered.db"))

You should see:

<!-- expect:ex3-tamper -->
```text
refund result 99a3838a: cents 4200 -> 42
FAIL 99a3838a...: result digest does not match stored result
verify exit code: 1
```

The edit left the node ID alone and changed the stored result, so the result no longer matches its digest. The `...` stands for the rest of the node ID.

## Exercise 4: Replay

**Goal:** run the agent again with nothing live behind it.

In `replay` mode the runtime serves recorded steps and refuses everything else. Look at the first three lines of `gated_agent`: in replay, the model function and every tool handler are replaced by a sentinel that raises if it is ever called.

1. Run the cell as it is, in `hybrid` mode. Then set `MODE = "replay"` and run it again.

In [ ]:
MODE = "hybrid"      # run once as it is, then change to "replay"

_ = gated_agent(8, mode=MODE)
root = root_id_by_hand()
print("seal", evidence(root)[1][:16])

In `hybrid` mode the runtime would have to execute the refund again, so the policy applies, and the approval is used. That is the refusal from the end of Exercise 2.

In `replay` mode nothing can execute. The record is served as it happened, the approved refund included, with no live calls and the same seal as before:

<!-- expect:ex4-replay -->
```text
  tool lookup_order -> status delivered
  tool issue_refund -> refund_id R-77, cents 4200
  FINAL: Refund of $42.00 issued for order A-1001.
  live calls: none
```

2. Change one word of the input.

In [ ]:
_ = gated_agent(8, mode="replay", task=TASK.replace("damaged", "late"))

<!-- expect:ex4-miss -->
```text
  MissingRecording: missing recording for model_call model=scripted-model digest
  live calls: none
```

A different input is a different node ID, and no recording exists under it. Replay stops. It does not go live.

3. Replay the edited copy from Exercise 3.

In [ ]:
try:
    _ = gated_agent(8, mode="replay", db="tampered.db")
except IntegrityError as detected:
    print("IntegrityError:", detected)

<!-- expect:ex4-tampered -->
```text
  tool lookup_order -> status delivered
IntegrityError: recording integrity check failed: 99a3838a...: result digest does not match stored result
```

Replay checks each recorded result against its digest before serving it, so the edited refund is never returned to the agent.

## The same flow on a real model

Optional. These cells run the gated flow once on a real model, in a separate store, with `agent()`, the finished loop from `workshop_kit.py`. They make three model calls in total. If the chosen path is not available, they run on the scripted model and say so.

In [ ]:
REAL_PATH = "colab"          # or "endpoint"
REAL_DB = "real-model.db"
REVIEWER = globals().get("REVIEWER", "your-name")

real = choose_model(REAL_PATH)
reset(REAL_DB)
paused = agent(REAL_DB, steps=8, gate=True, model=real)

In [ ]:
if paused["request"]:
    print("decision node", approve(paused["request"], REVIEWER, db=REAL_DB))
    done = agent(REAL_DB, steps=8, gate=True, model=real)
else:
    print("No approval request: the model did not ask for a refund. Status:", paused["status"])

In [ ]:
replayed = agent(REAL_DB, steps=8, gate=True, mode="replay", model=real)
print("verified, seal, nodes:", evidence(replayed["root"], REAL_DB))

The root ID is `e7c06c72` again, because it depends only on the run label. Below the root the tree differs from the scripted one: the model's own words are part of each later step's input, so they are part of each later node's ID. The replay made no model call.

## What you used

| You want | Call | It says no with |
|---|---|---|
| A governed model call | `run.model_call(payload, fn=...)` | |
| A governed tool call | `run.tool_call(name, args)` | `PolicyViolation` for an unknown tool or bad arguments |
| A hard limit | `runtime.run(label, budget=Budget(steps=..., tokens=..., usd=...))` | `BudgetExceeded` |
| A named approval | `ApprovalPolicy(store)`, `request_approval(run, name, args)`, `decide_approval(store, request, approved=..., reviewer=...)` | `PolicyViolation` |
| The record | `pollard show`, `pollard report`, `pollard verify`, `pollard seal` | exit code 1 from `verify` |
| No live calls | `Runtime(store, mode="replay")` | `MissingRecording`, `IntegrityError` |

The three modes: `record` executes every step and records it. `hybrid` serves a recorded step and executes a missing one. `replay` serves recorded steps and nothing else.

Every refusal carries a `refusal_id`, the ID of the refusal node in the tree.

## Gallery: every example in the Pollard repository

The rest of this notebook runs the examples, recipes and experiment scripts that ship with Pollard, from a copy of the repository at tag `v1.6.2`. Run the next cell once, then any cell below it in any order.

The nineteen walkthroughs need no key and no network, except `09`, which talks to a database or broker you supply. Numbers they print for timing and storage are local diagnostics, not published measurements.

| Goes with | Walkthroughs |
|---|---|
| Exercise 1, budgets | `01`, `03`, `16` |
| Exercise 2, gates | `04`, `12`, `18` |
| Exercise 3, evidence | `11` |
| Exercise 4, replay | `05`, `10`, `14` |
| Take home | `02`, `06`, `07`, `08`, `09`, `13`, `15`, `17`, `19` |

In [ ]:
POLLARD_SRC = os.path.join("scratch", f"pollard-v{POLLARD_VERSION}")
if not os.path.isdir(POLLARD_SRC):
    subprocess.run(["git", "-c", "advice.detachedHead=false", "clone", "-q", "--depth", "1",
                    "--branch", f"v{POLLARD_VERSION}", "https://github.com/jemsbhai/pollard", POLLARD_SRC], check=True)


def run_script(path, *arguments, tail=None):
    # Runs one script from the root of the Pollard repository and prints what it prints, or only its last lines.
    done = subprocess.run([sys.executable, path, *arguments], cwd=POLLARD_SRC, capture_output=True, text=True)
    lines = (done.stdout + done.stderr).rstrip().splitlines()
    if tail and len(lines) > tail:
        print(f"... {len(lines) - tail} lines not shown; pass tail=None to see them")
        lines = lines[-tail:]
    print("\n".join(lines))
    if done.returncode != 0:
        print(f"[exit code {done.returncode}]")


def run_example(name, *arguments, tail=None):
    run_script(os.path.join("examples", name), *arguments, tail=tail)


def run_recipe(name, *arguments, tail=None):
    # Each run gets a fresh database. A recipe records in "record" mode, where a second run on the same
    # store executes every step again and spends the recipe's own budget a second time.
    database = name.replace(".py", ".db")
    reset(os.path.join(POLLARD_SRC, database))
    run_script(os.path.join("docs", "recipes", name), "--database", database, *arguments, tail=tail)


print("Pollard repository at", POLLARD_SRC)

In [ ]:
# 01: One model-shaped call is metered and recorded as a node.
run_example("01_governed_call.py")

In [ ]:
# 02: Four branches share a parent, a local judge selects one, and losing tips are pruned.
run_example("02_best_of_n.py")

In [ ]:
# 03: An exact zero-step budget refuses before the model function runs.
run_example("03_budget_stop.py")

In [ ]:
# 04: An unregistered action fails closed before any handler executes.
run_example("04_firewall.py")

In [ ]:
# 05: Strict replay serves a committed result while a sentinel live client remains unreachable.
run_example("05_replay_ci.py")

In [ ]:
# 06: Deterministic mock checks for shared prefixes, budget refusal, and registry refusal.
run_example("06_phase4_benchmarks.py")

In [ ]:
# 07: SQLite payload interning preserves identity while changing file growth in a synthetic workload. Takes about five seconds.
run_example("07_phase7_storage.py")

In [ ]:
# 08: A resumed SQLite request window refuses the next call, and two disconnected stores merge cleanly.
run_example("08_phase8_scaleout.py")

**`09_distributed_stores.py`** performs one deterministic record, strict replay, tree verification and seal against PostgreSQL, Redis, MongoDB, Neo4j or Kafka. It makes no model request, but it needs a backend you can reach, so it is not run here. With Redis, for example:

```python
%pip install -q "pollard[redis]=={POLLARD_VERSION}"
os.environ["POLLARD_REDIS_URL"] = "rediss://<user>:<password>@<host>:6379/0"
run_example("09_distributed_stores.py", "--backend", "redis", "--store-id", "workshop")
```

The other backends and their environment variables are listed in the [examples README](https://github.com/jemsbhai/pollard/blob/v1.6.2/examples/README.md).

In [ ]:
# 10: Retained stream chunks are emitted live and again during strict replay without calling the model function.
run_example("10_streaming_replay.py")

In [ ]:
# 11: A schema-marked secret reaches its handler but only a deterministic redaction marker reaches the audit tree.
run_example("11_sensitive_fields.py")

In [ ]:
# 12: A side effect is suppressed during preview and paused for an explicit confirmation token before execution.
run_example("12_dry_run_confirmation.py")

In [ ]:
# 13: Async model and tool functions share the same governed budget and audit tree.
run_example("13_async_workflow.py")

In [ ]:
# 14: A golden result is compared with a separately stored live observation without changing the recording.
run_example("14_revalidate_recording.py")

In [ ]:
# 15: Planner, three specialists, and reviewer have separate identities, inherited limits, tool permissions, and recorded dependencies.
run_example("15_agent_team.py")

In [ ]:
# 16: Four workers attach JSON contexts through independent SQLite connections and share an exact three-step cap.
run_example("16_distributed_team.py")

In [ ]:
# 17: One worker and three workers perform the same deterministic task under recorded accounting.
run_example("17_team_comparison.py")

In [ ]:
# 18: Exact tool approval and checkpoint survive reconstruction of the worker runtime.
run_example("18_durable_approval.py")

In [ ]:
# 19: Synthetic Codex result recording and strict replay without launching a model.
run_example("19_codex_recording.py")

## Gallery: integration recipes

The twelve recipes in [`docs/recipes`](https://github.com/jemsbhai/pollard/blob/v1.6.2/docs/recipes/README.md) place Pollard around an existing provider client, framework workflow, graph node or MCP session. Five of them run offline by default. The next cell installs the optional packages those five need; it can take a minute.

In [ ]:
%pip install -q "pollard[pydantic,pydantic-ai,langchain,mcp]=={POLLARD_VERSION}"

In [ ]:
# Pydantic-generated action schema, preview, confirmation, redaction, and replay.
run_recipe("pydantic_refund_workflow.py")

In [ ]:
# Typed claim triage with every pydantic-ai model request and registered tool execution governed.
run_recipe("pydantic_ai_claim_triage.py")

In [ ]:
# Local return-policy RAG through a registered LangChain retriever and typed answer.
run_recipe("langchain_support_rag.py")

In [ ]:
# Typed LangChain incident triage with governed model and action boundaries.
run_recipe("langchain_incident_response.py")

In [ ]:
# MCP tool discovery and invocation through the registry firewall, against a local stdio server.
run_recipe("mcp_registry.py", "--stdio", "--server-arg", os.path.join("examples", "mcp_demo_server.py"),
           sys.executable, "search", '{"query": "pollard"}')

The other seven recipes make live requests to a provider, so they need a credential and are not run here. Read each script and set a spending limit on the provider side before you run one. From the root of the Pollard repository:

| Recipe | Boundary it demonstrates | Install | Credential or endpoint |
|---|---|---|---|
| [`openai_tool_loop.py`](https://github.com/jemsbhai/pollard/blob/v1.6.2/docs/recipes/openai_tool_loop.py) | OpenAI Responses tool calls through a Pollard registry | `pollard[openai]` | `OPENAI_API_KEY` |
| [`anthropic_tool_loop.py`](https://github.com/jemsbhai/pollard/blob/v1.6.2/docs/recipes/anthropic_tool_loop.py) | Anthropic Messages tool use plus token precheck | `pollard[anthropic]` | `ANTHROPIC_API_KEY` |
| [`azure_openai.py`](https://github.com/jemsbhai/pollard/blob/v1.6.2/docs/recipes/azure_openai.py) | Azure OpenAI v1 through the OpenAI Responses adapter | `pollard[azure-openai]` | Endpoint and deployment, plus an API key or DefaultAzureCredential |
| [`bedrock_converse.py`](https://github.com/jemsbhai/pollard/blob/v1.6.2/docs/recipes/bedrock_converse.py) | Amazon Bedrock Converse through a boto3 client | `pollard[bedrock]` | AWS credential chain, Region, model access |
| [`litellm_cloud.py`](https://github.com/jemsbhai/pollard/blob/v1.6.2/docs/recipes/litellm_cloud.py) | Vertex AI, Azure AI, SageMaker, OCI, Watsonx, Databricks, Bedrock or another LiteLLM route | `pollard[litellm]` | The selected LiteLLM provider configuration |
| [`langgraph_node.py`](https://github.com/jemsbhai/pollard/blob/v1.6.2/docs/recipes/langgraph_node.py) | Pollard inside one LangGraph node | `pollard[langgraph]` | `OPENAI_API_KEY` |
| [`pydantic_ai_wrap.py`](https://github.com/jemsbhai/pollard/blob/v1.6.2/docs/recipes/pydantic_ai_wrap.py) | One complete pydantic-ai run as a Pollard step | `pollard[pydantic-ai]` | `OPENAI_API_KEY` |

For example: `python docs/recipes/openai_tool_loop.py`, after `pip install "pollard[openai]==1.6.2"` and with `OPENAI_API_KEY` set.

## Gallery: experiment scripts

These scripts run Pollard's formal evidence protocols. Their recorded results ship in the repository's `evidence` folder. Three of them run offline here. The first replays the recorded EXP-006 case studies without calling a model. The other two print numbers from this machine, not the recorded results.

In [ ]:
# Verifies and replays the three recorded EXP-006 case studies. No model call, no network.
run_example("exp_006_verify.py")

In [ ]:
# EXP-007 in a small configuration: per-step overhead on this machine. Its output ends with the run's status.
run_example("exp_007_overhead.py", "--steps", "4", "--samples", "3", "--warmups", "0", tail=14)

In [ ]:
# EXP-004, the SQLite storage-curve experiment. Takes about half a minute. Its output ends with the fitted exponents.
run_example("exp_004_storage.py", tail=24)

The rest need hardware or services that a notebook does not have:

| Script | What it records | Needs |
|---|---|---|
| [`exp_001_local_model.py`](https://github.com/jemsbhai/pollard/blob/v1.6.2/examples/exp_001_local_model.py) | The EXP-001 shared-prefix experiment on a local model | An NVIDIA GPU, a pinned llama.cpp server and a local GGUF model |
| [`exp_005_contention.py`](https://github.com/jemsbhai/pollard/blob/v1.6.2/examples/exp_005_contention.py) | EXP-005 contention and estimator-bound evidence | At least two PostgreSQL targets |
| [`exp_006_research.py`](https://github.com/jemsbhai/pollard/blob/v1.6.2/examples/exp_006_research.py) | EXP-006A: research and summarize a fixed offline document set | A pinned llama.cpp server and a local model |
| [`exp_006_code_fix.py`](https://github.com/jemsbhai/pollard/blob/v1.6.2/examples/exp_006_code_fix.py) | EXP-006B: fix a bug in a pinned repository with pinned tests | A pinned llama.cpp server and a local model |
| [`exp_006_mcp_household.py`](https://github.com/jemsbhai/pollard/blob/v1.6.2/examples/exp_006_mcp_household.py) | EXP-006C: household planning across three pinned MCP servers | A pinned llama.cpp server, a local model and `pollard[mcp]` |

Their commands are in the [examples README](https://github.com/jemsbhai/pollard/blob/v1.6.2/examples/README.md), and the recorded results are in the [evidence index](https://github.com/jemsbhai/pollard/blob/v1.6.2/evidence/README.md).

## Where to go next

- Pollard: <https://github.com/jemsbhai/pollard>, or `pip install pollard`
- Paper: Syed and Silaghi, "Pollard: Governed Semantic Execution Trees for Framework-Independent AI Agent Control and Replay", IEEE ICTAI 2026, <https://arxiv.org/abs/2610.16145>
- This workshop: <https://github.com/jemsbhai/devfestTampaPollard2026>